In [0]:
# =========================================================
# NOTEBOOK : 01 - BRONZE LAYER
# PURPOSE  : Incremental Ingestion using Auto Loader
# =========================================================

# =========================================================
# 1. IMPORTS
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col
import logging

# =========================================================
# 2. LOGGING
# =========================================================

logger = logging.getLogger("ecommerce_pipeline.bronze")

# =========================================================
# 3. PATHS (Databricks me dbutils.widgets se bhi le sakte ho)
# =========================================================

RAW_PATH          = "/Volumes/workspace/default/sales_volume/row__sales_data/"
SCHEMA_PATH       = "/Volumes/workspace/default/sales_volume/schema"
BRONZE_PATH       = "/Volumes/workspace/default/sales_volume/bronze/data"
BRONZE_CHECKPOINT = "/Volumes/workspace/default/sales_volume/bronze/checkpoint"

# =========================================================
# 4. BRONZE FUNCTION
# =========================================================

def run_bronze(spark):

    try:

        logger.info("Starting Bronze Layer")

        # --------------------------------------------------
        # Auto Loader se CSV read karo (Incremental)
        # --------------------------------------------------

        bronze_df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.schemaLocation", SCHEMA_PATH)
            .load(RAW_PATH)
        )

        # --------------------------------------------------
        # Audit Columns Add karo
        # --------------------------------------------------

        bronze_df = bronze_df \
            .withColumn("ingestion_time", current_timestamp()) \
            .withColumn("source_file", col("_metadata.file_path"))

        # --------------------------------------------------
        # Delta me Incremental Write karo
        # --------------------------------------------------

        bronze_query = (
            bronze_df.writeStream
            .format("delta")
            .option("checkpointLocation", BRONZE_CHECKPOINT)
            .outputMode("append")
            .trigger(availableNow=True)
            .start(BRONZE_PATH)
        )

        bronze_query.awaitTermination()

        logger.info("Bronze Layer Completed Successfully")

    except Exception as e:

        logger.error(f"Bronze Layer Failed : {str(e)}")
        raise


# =========================================================
# 5. DIRECT RUN (Agar ye notebook akela chalao)
# =========================================================

if __name__ == "__main__":

    spark = SparkSession.builder \
        .appName("Bronze-Layer") \
        .config("spark.sql.shuffle.partitions", "8") \
        .config("spark.databricks.delta.optimizeWrite.enabled", "true") \
        .config("spark.databricks.delta.autoCompact.enabled", "true") \
        .getOrCreate()

    run_bronze(spark)

In [0]:
# =========================================================
# NOTEBOOK : 01 - BRONZE LAYER
# PURPOSE  : Incremental Ingestion using Auto Loader
# =========================================================

# =========================================================
# 1. IMPORTS
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col
import logging

# =========================================================
# 2. LOGGING
# =========================================================

logger = logging.getLogger("ecommerce_pipeline.bronze")

# =========================================================
# 3. PATHS (Databricks me dbutils.widgets se bhi le sakte ho)
# =========================================================

RAW_PATH          = "/Volumes/workspace/default/sales_volume/row__sales_data/"
SCHEMA_PATH       = "/Volumes/workspace/default/sales_volume/schema"
BRONZE_PATH       = "/Volumes/workspace/default/sales_volume/bronze/data"
BRONZE_CHECKPOINT = "/Volumes/workspace/default/sales_volume/bronze/checkpoint"

# =========================================================
# 4. BRONZE FUNCTION
# =========================================================

def run_bronze(spark):

    try:

        logger.info("Starting Bronze Layer")

        # --------------------------------------------------
        # Auto Loader se CSV read karo (Incremental)
        # --------------------------------------------------

        bronze_df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.schemaLocation", SCHEMA_PATH)
            .load(RAW_PATH)
        )

        # --------------------------------------------------
        # Audit Columns Add karo
        # --------------------------------------------------

        bronze_df = bronze_df \
            .withColumn("ingestion_time", current_timestamp()) \
            .withColumn("source_file", col("_metadata.file_path"))

        # --------------------------------------------------
        # Delta me Incremental Write karo
        # --------------------------------------------------

        bronze_query = (
            bronze_df.writeStream
            .format("delta")
            .option("checkpointLocation", BRONZE_CHECKPOINT)
            .outputMode("append")
            .trigger(availableNow=True)
            .start(BRONZE_PATH)
        )

        bronze_query.awaitTermination()

        logger.info("Bronze Layer Completed Successfully")

    except Exception as e:

        logger.error(f"Bronze Layer Failed : {str(e)}")
        raise


# =========================================================
# 5. DIRECT RUN (Agar ye notebook akela chalao)
# =========================================================

if __name__ == "__main__":

    spark = SparkSession.builder \
        .appName("Bronze-Layer") \
        .config("spark.sql.shuffle.partitions", "8") \
        .config("spark.databricks.delta.optimizeWrite.enabled", "true") \
        .config("spark.databricks.delta.autoCompact.enabled", "true") \
        .getOrCreate()

    run_bronze(spark)